In [1]:
### Magics
%reload_ext autoreload
%autoreload 2

### Imports
import copy, itertools, functools, numpy as np, pandas as pd, time, requests, io
import re, os, sys, json, pickle, tqdm, collections, random, warnings, pprint, typing
from scipy import stats

### Matplotlib
%config InlineBackend.figure_formats = ["svg"]
%matplotlib inline

import matplotlib.pyplot as plt
from matplotlib import rcParams, figure
rcParams["font.family"] = "Fira Code"
# rcParams["font.family"] = "Palatino"
rcParams["axes.titleweight"] = "bold"
rcParams["axes.labelsize"] = "large"

### Copy and Paste Tools
# Skip cell: %%script false --no-raise-error

In [ ]:
input_seq_id_to_seq: dict[str, str] = {}
for filename in [
    "/people/druc594/containers/python-tunneller.sif/home/tunneller/Snekmer/execution/input/all_pos_6664.fasta",
    "/people/druc594/containers/python-tunneller.sif/home/tunneller/Snekmer/execution/input/all_neg_106757.fasta",
]:
    with open(filename, "r") as f:
        for line in f:
            if line.startswith(">"):
                input_seq_id = line.strip().split(">")[-1]
            else:
                input_seq = line.strip()
                input_seq_id_to_seq[input_seq_id] = input_seq

X = np.array(list(input_seq_id_to_seq.values()))
X_lab = np.array(list(input_seq_id_to_seq.keys()))
y = np.array([0 if "neg" in seq_id.lower() else 1 for seq_id in input_seq_id_to_seq.keys()])

In [3]:
assert X.shape[0] == y.shape[0]

In [36]:
from sklearn.model_selection import StratifiedKFold
import yaml

os.chdir("/Users/druc594/Desktop/SnekmerClean/editables/Snekmer/")
N_FOLDS = 4

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

inputs_dir = "../../skf/input"
annotation_dir = "../../skf/annotations"
num_neg = -1
num_pos = -1
info_str_short = ""
for fold, (train_indexes, test_indexes) in tqdm.tqdm(enumerate(skf.split(X, y)), total=N_FOLDS):
    for tr_te, idx_set in zip(["train", "test"], [train_indexes, test_indexes]):
        for class_, cls_ in zip([0, 1], ["Negative", "Positive"]):
            temp_filename = os.path.join(inputs_dir, "temp.fasta")
            with open(temp_filename, "w") as f:
                n_seqs = 0
                for idx in idx_set:
                    if y[idx] == class_:
                        f.write(f">{X_lab[idx]}\n{X[idx]}\n")
                        n_seqs += 1

            info_str_short = f"f{fold}trte{tr_te}"
            info_str = f"{cls_}_{n_seqs}"
            final_filename = os.path.join(inputs_dir, f"{info_str}.{info_str_short}")
            os.rename(temp_filename, final_filename)

            if cls_ == "Positive":
                num_pos = n_seqs

                with open("../../skf/base_config.yaml", "r") as f:
                    config = yaml.safe_load(f)
                    config["input_file_exts"] = [f"{info_str_short}"]
                    config["input_file_regex"] = ".*"
                with open(os.path.join("../../skf/configs", f"{info_str_short}.yaml"), "w") as f:
                    yaml.dump(config, f)


            else:
                num_neg = n_seqs

100%|██████████| 4/4 [00:06<00:00,  1.57s/it]


In [ ]:
os.chdir("/Users/druc594/Desktop/SnekmerClean/skf")

import shutil

def copy_folders(sub_name: str):
    shutil.copytree(
        os.path.join("output", "apply_inputs", sub_name),
        os.path.join(sub_name),
        dirs_exist_ok=True,
    )


result_dfs = {}
for fold in tqdm.tqdm(range(N_FOLDS)):
    train_cmd = f"snekmer learn --configfile configs/f{fold}trtetrain.yaml"
    print(train_cmd)
    subprocess.run(train_cmd, shell=True)

    [copy_folders(sub_name) for sub_name in ["confidence", "counts", "stats"]]

    test_cmd = f"snekmer apply --configfile configs/f{fold}trtetest.yaml"
    print(test_cmd)
    p = subprocess.run(test_cmd, shell=True, capture_output=True)
    if p.returncode != 0:
        raise RuntimeError(f"Error in fold {fold}: {p.stderr.decode('utf-8')}")


    result_df = {
        os.path.join("output", "apply", x): pd.read_csv(os.path.join("output", "apply", x))
        for x in os.listdir("output/apply")
        if x.endswith(".csv")
    }

    result_dfs |= {fold: result_df}

    os.remove(os.path.join("output"))

  0%|          | 0/4 [00:00<?, ?it/s]

snekmer learn --configfile configs/f0trtetrain.yaml


Building DAG of jobs...
The params used to generate one or several output files has changed:
    To inspect which output files have changes, run 'snakemake --list-params-changes'.
    To trigger a re-run, use 'snakemake -R $(snakemake --list-params-changes)'.
Using shell: /bin/bash
Provided cores: 10
Rules claiming more threads will be scaled down.
Job stats:
job                         count    min threads    max threads
------------------------  -------  -------------  -------------
all                             1              1              1
copy_results_for_apply          1              1              1
eval_apply_reverse_seqs         2              1              1
eval_apply_sequences            2              1              1
evaluate                        1              1              1
learn                           2              1              1
merge                           1              1              1
reverseDecoy_evaluations        1              1              

Dataframes merged: 0 out of 2
Dataframes merged: 1 out of 2

Checking for base file to merge with.

No file type detected. Please use a .csv file in input/base directory.


Database Merged. Not merged with base file.



[Wed May  7 02:02:13 2025]
Finished job 5.
5 of 13 steps (38%) done
Select jobs to execute...

[Wed May  7 02:02:13 2025]
rule eval_apply_reverse_seqs:
    input: output/vector/Positive_4998.npz, annotations/annots.ann, output/learn/kmer-counts-total.csv
    output: output/eval_apply_reversed/seq-annotation-scores-Positive_4998.csv.gz
    log: output/eval_apply_reversed/log/Positive_4998.log
    jobid: 9
    wildcards: nb=Positive_4998
    resources: tmpdir=/var/folders/ky/bjjbcg6j4_15tx4wc_zd9pd40000gn/T

[Wed May  7 02:02:13 2025]
rule eval_apply_reverse_seqs:
    input: output/vector/Negative_2499.npz, annotations/annots.ann, output/learn/kmer-counts-total.csv
    output: output/eval_apply_reversed/seq-annotation-scores-Negative_2499.csv.gz
    log: output/eval_apply_reversed/log/Negative_2499.log
    jobid: 10
    wildcards: nb=Negative_2499
    resources: tmpdir=/var/folders/ky/bjjbcg6j4_15tx4wc_zd9pd40000gn/T

[Wed May  7 02:02:13 2025]
rule eval_apply_sequences:
    input: outpu

...

 75%|███████▌  | 3/4 [04:21<01:00, 60.61s/it] 

snekmer learn --configfile configs/f3trtetrain.yaml


Building DAG of jobs...
The input used to generate one or several output files has changed:
    To inspect which output files have changes, run 'snakemake --list-input-changes'.
    To trigger a re-run, use 'snakemake -R $(snakemake --list-input-changes)'.
Nothing to be done (all requested files are present and up to date).
The input used to generate one or several output files has changed:
    To inspect which output files have changes, run 'snakemake --list-input-changes'.
    To trigger a re-run, use 'snakemake -R $(snakemake --list-input-changes)'.
Complete log: /Users/druc594/Desktop/SnekmerClean/skf/.snakemake/log/2025-05-07T020419.644357.snakemake.log


snekmer apply --configfile configs/f3trtetest.yaml


100%|██████████| 4/4 [04:28<00:00, 67.07s/it]


In [31]:
result_dfs[0]['output/apply/kmer-summary-Negative_833.csv']

,Sequence,Prediction,Score,delta,Confidence
0,|Negative_1|,NaN,NaN,0.00,0.449612
1,|Negative_3|,NEG,0.802062,0.09,0.517375
2,|Negative_11|,NaN,NaN,0.00,0.449612
3,|Negative_26|,NEG,0.769086,0.06,0.554217
4,|Negative_30|,POS,0.763473,0.02,0.482625
...,...,...,...,...,...
828,|Negative_3316|,NaN,NaN,0.00,0.449612
829,|Negative_3324|,NEG,0.757794,0.05,0.646840
830,|Negative_3327|,NaN,NaN,0.00,0.449612
831,|Negative_3329|,NEG,0.823231,0.03,0.427007


In [48]:
result_dfs[1]

{'output/apply/kmer-summary-Negative_833.csv':             Sequence Prediction     Score  delta  Confidence
 0       |Negative_1|        NaN       NaN   0.00    0.449612
 1       |Negative_3|        NEG  0.802062   0.09    0.517375
 2      |Negative_11|        NaN       NaN   0.00    0.449612
 3      |Negative_26|        NEG  0.769086   0.06    0.554217
 4      |Negative_30|        POS  0.763473   0.02    0.482625
 ..               ...        ...       ...    ...         ...
 828  |Negative_3316|        NaN       NaN   0.00    0.449612
 829  |Negative_3324|        NEG  0.757794   0.05    0.646840
 830  |Negative_3327|        NaN       NaN   0.00    0.449612
 831  |Negative_3329|        NEG  0.823231   0.03    0.427007
 832  |Negative_3330|        NEG  0.745028   0.03    0.427007
 
 [833 rows x 5 columns],
 'output/apply/kmer-summary-Positive_1666.csv':              Sequence Prediction     Score  delta  Confidence
 0        |Positive_0|        NEG  0.731483   0.02    0.482625
 1       |